## Retail Dataset Generator

Generates a synthetic **retail** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `retail` | `customers` | ~5K | `orders` | 100K-500K | 60+ products across 7 categories, demographics, product-level return rates |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `retail` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.retail') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.retail');

In [0]:
%pip install faker --quiet

In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, DateType, StringType

fake = Faker()
Faker.seed(77)
random.seed(77)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "retail"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"
NOW = datetime(2026, 3, 21)

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

# --- Customers table (~5,000 rows) ---
regions = ["Northeast", "Southeast", "Midwest", "Southwest", "West Coast", "Pacific Northwest", "Mountain West"]
cities = {
    "Northeast": ["New York", "Boston", "Philadelphia", "Pittsburgh", "Hartford"],
    "Southeast": ["Miami", "Atlanta", "Charlotte", "Nashville", "Orlando", "Raleigh"],
    "Midwest": ["Chicago", "Detroit", "Minneapolis", "Columbus", "Indianapolis", "Milwaukee"],
    "Southwest": ["Dallas", "Phoenix", "Houston", "San Antonio", "Austin", "Tucson"],
    "West Coast": ["Los Angeles", "San Francisco", "Seattle", "San Diego", "Sacramento"],
    "Pacific Northwest": ["Portland", "Seattle", "Boise", "Eugene", "Spokane"],
    "Mountain West": ["Denver", "Salt Lake City", "Albuquerque", "Las Vegas", "Boise"]}
tiers = ["Bronze", "Silver", "Gold", "Platinum"]

NUM_CUSTOMERS = 5000
customers = []
for i in range(1, NUM_CUSTOMERS + 1):
    region = random.choice(regions)
    signup = date(2019, 1, 1) + timedelta(days=random.randint(0, 2500))
    tenure_years = (NOW.date() - signup).days / 365.25
    age = int(clamp(random.gauss(38, 14), 18, 85))
    dob = date(2026, 3, 20) - timedelta(days=int(age * 365.25) + random.randint(0, 364))
    if tenure_years > 5:
        tier = random.choices(tiers, weights=[15, 25, 35, 25])[0]
    elif tenure_years > 3:
        tier = random.choices(tiers, weights=[25, 35, 25, 15])[0]
    elif tenure_years > 1:
        tier = random.choices(tiers, weights=[35, 35, 20, 10])[0]
    else:
        tier = random.choices(tiers, weights=[60, 25, 10, 5])[0]
    customers.append(Row(
        customer_id=i,
        customer_name=fake.name(),
        email=fake.email(),
        date_of_birth=dob,
        age=age,
        region=region,
        city=random.choice(cities[region]),
        loyalty_tier=tier,
        signup_date=signup,
        is_active=random.choices([True, False], weights=[85, 15])[0]
    ))

customer_lookup = {c.customer_id: c for c in customers}

customers_df = spark.createDataFrame(customers)
customers_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.customers")
print(f"✔ Created {CATALOG_SCHEMA}.customers ({customers_df.count()} rows)")

# --- Orders table (randomized ~100K-500K rows) ---
product_catalog = [
    ("Electronics", ["Wireless Earbuds", "Bluetooth Speaker", "Phone Case", "USB-C Cable", "Smart Watch",
                     "Tablet Stand", "Portable Charger", "Webcam", "Keyboard", "Mouse Pad"]),
    ("Clothing",    ["T-Shirt", "Jeans", "Sneakers", "Jacket", "Hoodie", "Cap",
                     "Dress Shirt", "Socks Pack", "Scarf", "Belt"]),
    ("Home & Kitchen", ["Coffee Maker", "Blender", "Throw Pillow", "Candle Set", "Cutting Board",
                        "Water Bottle", "Air Fryer", "Dutch Oven", "Wine Glasses", "Dish Towels"]),
    ("Beauty",     ["Moisturizer", "Sunscreen", "Lip Balm", "Shampoo", "Face Mask",
                    "Perfume", "Hair Dryer", "Nail Kit"]),
    ("Sports",     ["Yoga Mat", "Dumbbells", "Resistance Band", "Running Shoes",
                    "Gym Bag", "Jump Rope", "Foam Roller", "Protein Shaker"]),
    ("Books & Media", ["Bestseller Novel", "Cookbook", "Self-Help Book", "Vinyl Record",
                       "Board Game", "Puzzle Set"]),
    ("Grocery",    ["Organic Coffee", "Trail Mix", "Protein Bars", "Olive Oil",
                    "Sparkling Water", "Dark Chocolate", "Honey", "Dried Fruit"]),
]
category_price_params = {
    "Electronics":    (3.8, 0.7, 5.99, 499.99),
    "Clothing":       (3.3, 0.6, 9.99, 199.99),
    "Home & Kitchen": (3.2, 0.8, 7.99, 299.99),
    "Beauty":         (2.7, 0.5, 4.99, 89.99),
    "Sports":         (3.4, 0.7, 9.99, 249.99),
    "Books & Media":  (2.6, 0.4, 6.99, 49.99),
    "Grocery":        (2.0, 0.4, 2.99, 39.99),
}

channel_payment_weights = {
    "Website":    [35, 15, 30, 10, 10],
    "Mobile App": [20, 10, 15, 45, 10],
    "In-Store":   [40, 35, 0, 15, 10],
}
payment_methods = ["Credit Card", "Debit Card", "PayPal", "Apple Pay", "Gift Card"]

tier_discount_weights = {
    "Bronze":   [55, 15, 12, 10, 5, 3],
    "Silver":   [40, 15, 15, 15, 10, 5],
    "Gold":     [25, 10, 18, 20, 17, 10],
    "Platinum": [15, 8, 15, 20, 22, 20],
}

tier_activity = {"Bronze": 1.0, "Silver": 1.8, "Gold": 3.0, "Platinum": 5.0}
customer_pool = []
for c in customers:
    if c.is_active:
        weight = tier_activity[c.loyalty_tier]
        customer_pool.extend([c.customer_id] * int(weight * 10))
    else:
        customer_pool.append(c.customer_id)

# Product-level return rates (instead of category-level)
product_return_rate = {
    "Wireless Earbuds": 0.20, "Bluetooth Speaker": 0.15, "Phone Case": 0.08, "USB-C Cable": 0.05,
    "Smart Watch": 0.22, "Tablet Stand": 0.06, "Portable Charger": 0.10, "Webcam": 0.12,
    "Keyboard": 0.08, "Mouse Pad": 0.03,
    "T-Shirt": 0.25, "Jeans": 0.30, "Sneakers": 0.28, "Jacket": 0.22, "Hoodie": 0.18,
    "Cap": 0.10, "Dress Shirt": 0.24, "Socks Pack": 0.05, "Scarf": 0.12, "Belt": 0.15,
    "Coffee Maker": 0.12, "Blender": 0.10, "Throw Pillow": 0.08, "Candle Set": 0.04,
    "Cutting Board": 0.03, "Water Bottle": 0.05, "Air Fryer": 0.14, "Dutch Oven": 0.06,
    "Wine Glasses": 0.08, "Dish Towels": 0.03,
    "Moisturizer": 0.06, "Sunscreen": 0.04, "Lip Balm": 0.02, "Shampoo": 0.05,
    "Face Mask": 0.07, "Perfume": 0.10, "Hair Dryer": 0.12, "Nail Kit": 0.06,
    "Yoga Mat": 0.06, "Dumbbells": 0.04, "Resistance Band": 0.05, "Running Shoes": 0.20,
    "Gym Bag": 0.05, "Jump Rope": 0.03, "Foam Roller": 0.04, "Protein Shaker": 0.05,
    "Bestseller Novel": 0.02, "Cookbook": 0.03, "Self-Help Book": 0.04, "Vinyl Record": 0.05,
    "Board Game": 0.06, "Puzzle Set": 0.04,
    "Organic Coffee": 0.03, "Trail Mix": 0.02, "Protein Bars": 0.03, "Olive Oil": 0.02,
    "Sparkling Water": 0.01, "Dark Chocolate": 0.02, "Honey": 0.02, "Dried Fruit": 0.02,
}

return_reasons = ["Wrong size", "Defective product", "Not as described", "Changed mind",
                  "Better price elsewhere", "Arrived damaged", "Wrong item received"]

# --- dim_product (normalized product dimension) ---
dim_product_rows = []
product_name_to_id = {}
_product_id = 1
for _category, _products in product_catalog:
    _mu, _sigma, _lo, _hi = category_price_params[_category]
    _base_price = round(clamp(math.exp(_mu), _lo, _hi), 2)
    for _pname in _products:
        _rr = product_return_rate.get(_pname, 0.08)
        dim_product_rows.append(Row(
            product_id=_product_id,
            product_name=_pname,
            category=_category,
            base_price=_base_price,
            return_rate=_rr,
        ))
        product_name_to_id[_pname] = _product_id
        _product_id += 1

dim_product_df = spark.createDataFrame(dim_product_rows)
dim_product_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.dim_product")
print(f"✔ Created {CATALOG_SCHEMA}.dim_product ({dim_product_df.count()} rows)")

orders = []
order_lines = []
returns = []
order_line_id_seq = 0
return_id_seq = 0
NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
for i in range(1, NUM_EVENT_RECORDS + 1):
    order_id = 5000 + i
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 730))
    month = order_date.month

    # Smooth seasonal category weighting using sine curves
    holiday_boost = max(0, math.sin((month - 9) * math.pi / 3))  # peaks Nov-Dec
    fitness_boost = max(0, math.sin((month - 11) * math.pi / 6))  # peaks Jan-Feb
    beauty_boost = max(0, math.sin((month - 12) * math.pi / 4))   # peaks Feb-May
    cat_weights = [
        20 + 15 * holiday_boost,   # Electronics
        20,                         # Clothing
        18,                         # Home & Kitchen
        15 + 10 * beauty_boost,    # Beauty
        15 + 15 * fitness_boost,   # Sports
        8,                          # Books & Media
        10,                         # Grocery
    ]

    cust_id = random.choice(customer_pool)
    cust = customer_lookup[cust_id]

    num_items = min(5, int(random.expovariate(0.8)) + 1)
    line_rows = []
    products_in_order = []
    for _ in range(num_items):
        category, products = random.choices(product_catalog, weights=cat_weights)[0]
        product = random.choice(products)
        products_in_order.append(product)
        quantity = min(10, int(random.expovariate(1.2)) + 1)
        mu, sigma, lo, hi = category_price_params[category]
        unit_price = round(clamp(random.lognormvariate(mu, sigma), lo, hi), 2)
        line_total = round(quantity * unit_price, 2)
        line_rows.append({
            "product_id": product_name_to_id[product],
            "product_name": product,
            "category": category,
            "quantity": quantity,
            "unit_price": unit_price,
            "line_total": line_total,
        })

    discount_pct = random.choices([0, 5, 10, 15, 20, 25],
                                  weights=tier_discount_weights[cust.loyalty_tier])[0]
    subtotal = sum(lr["line_total"] for lr in line_rows)
    total_amount = round(subtotal * (1 - discount_pct / 100), 2)

    channel = random.choices(["Website", "Mobile App", "In-Store"], weights=[35, 40, 25])[0]
    payment = random.choices(payment_methods, weights=channel_payment_weights[channel])[0]

    free_ship_prob = 0.15 + (0.1 if cust.loyalty_tier in ["Gold", "Platinum"] else 0) + (0.1 if total_amount > 75 else 0)
    if random.random() < free_ship_prob:
        ship_cost = 0.0
    else:
        ship_cost = round(clamp(random.lognormvariate(1.8, 0.5), 2.99, 24.99), 2)

    days_ago = (NOW.date() - order_date).days
    ret_rate = max(product_return_rate.get(p, 0.10) for p in products_in_order)
    if days_ago > 30:
        status = random.choices(["Delivered", "Shipped", "Processing", "Returned", "Cancelled"],
                                weights=[75 - ret_rate*100, 2, 0, ret_rate*100, 8])[0]
    elif days_ago > 7:
        status = random.choices(["Delivered", "Shipped", "Processing", "Returned", "Cancelled"],
                                weights=[55, 20, 5, ret_rate*50, 8])[0]
    else:
        status = random.choices(["Delivered", "Shipped", "Processing", "Returned", "Cancelled"],
                                weights=[10, 35, 40, 2, 13])[0]

    est_delivery = random.randint(1, 3) if channel == "In-Store" else random.randint(2, 14)

    orders.append(Row(
        order_id=order_id,
        customer_id=cust_id,
        order_date=order_date,
        channel=channel,
        payment_method=payment,
        order_status=status,
        discount_pct=discount_pct,
        total_amount=total_amount,
        shipping_cost=ship_cost,
        estimated_delivery_days=est_delivery,
    ))

    for lr in line_rows:
        order_line_id_seq += 1
        order_lines.append(Row(
            order_line_id=order_line_id_seq,
            order_id=order_id,
            product_id=lr["product_id"],
            product_name=lr["product_name"],
            category=lr["category"],
            quantity=lr["quantity"],
            unit_price=lr["unit_price"],
            line_total=lr["line_total"],
        ))

    if status == "Returned":
        return_id_seq += 1
        refund_amount = round(total_amount * random.uniform(0.8, 1.0), 2)
        return_date = order_date + timedelta(days=random.randint(3, 30))
        return_reason = random.choice(return_reasons)
        item_condition = random.choices(
            ["Like New", "Good", "Fair", "Damaged"],
            weights=[30, 35, 25, 10],
        )[0]
        if random.random() < 0.7:
            restocking_fee = 0.0
        else:
            restocking_fee = round(refund_amount * random.uniform(0.05, 0.15), 2)
        returns.append(Row(
            return_id=return_id_seq,
            order_id=order_id,
            return_date=return_date,
            refund_amount=refund_amount,
            return_reason=return_reason,
            item_condition=item_condition,
            restocking_fee=restocking_fee,
        ))

orders_df = spark.createDataFrame(orders)
orders_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.orders")
print(f"✔ Created {CATALOG_SCHEMA}.orders ({orders_df.count()} rows)")

order_lines_df = spark.createDataFrame(order_lines)
order_lines_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.order_lines")
print(f"✔ Created {CATALOG_SCHEMA}.order_lines ({order_lines_df.count()} rows)")

_returns_schema = StructType([
    StructField("return_id", IntegerType(), False),
    StructField("order_id", IntegerType(), False),
    StructField("return_date", DateType(), True),
    StructField("refund_amount", DoubleType(), True),
    StructField("return_reason", StringType(), True),
    StructField("item_condition", StringType(), True),
    StructField("restocking_fee", DoubleType(), True),
])
returns_df = spark.createDataFrame(returns, schema=_returns_schema) if returns else spark.createDataFrame([], _returns_schema)
returns_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.returns")
print(f"✔ Created {CATALOG_SCHEMA}.returns ({returns_df.count()} rows)")

print("\n--- Customers (sample) ---")
display(customers_df.limit(5))
print("\n--- dim_product (sample) ---")
display(dim_product_df.limit(5))
print("\n--- Orders (sample) ---")
display(orders_df.limit(5))
print("\n--- order_lines (sample) ---")
display(order_lines_df.limit(5))
print("\n--- returns (sample) ---")
display(returns_df.limit(5))

In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
    for col, comment in comments.items():
        spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
    print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.retail.customers", {
    "customer_id":   "Unique identifier for the customer",
    "customer_name": "Full name of the customer (generated via Faker)",
    "email":         "Customer email address (generated via Faker)",
    "date_of_birth": "Date of birth (DateType)",
    "age":           "Current age in years (normal distribution, mean 38)",
    "region":        "Geographic region (7 US regions)",
    "city":          "City within the region (4-6 cities per region)",
    "loyalty_tier":  "Loyalty program tier: Bronze, Silver, Gold, or Platinum",
    "signup_date":   "Date the customer registered (DateType)",
    "is_active":     "Whether the customer account is currently active",
})

apply_comments(f"{CATALOG}.retail.orders", {
    "order_id":                "Unique identifier for the order",
    "customer_id":             "Foreign key referencing customers.customer_id",
    "order_date":              "Date the order was placed (DateType)",
    "product_name":            "Name of the purchased product (60+ products)",
    "category":                "Product category: Electronics, Clothing, Home & Kitchen, Beauty, Sports, Books & Media, or Grocery",
    "quantity":                "Number of units ordered (exponential-based, most orders 1-2 items)",
    "unit_price":              "Price per unit in USD (log-normal per category)",
    "discount_pct":            "Discount percentage applied (0, 5, 10, 15, 20, or 25). Tier-weighted",
    "total_amount":            "Final order total after discount",
    "payment_method":          "Payment method: Credit Card, Debit Card, PayPal, Apple Pay, or Gift Card",
    "channel":                 "Sales channel: Website, Mobile App, or In-Store",
    "order_status":            "Order status: Delivered, Shipped, Processing, Returned, or Cancelled",
    "shipping_cost":           "Shipping cost in USD. 0.0 for free shipping",
    "estimated_delivery_days": "Estimated delivery time in days. 1-3 for In-Store, 2-14 for online",
    "return_reason":           "Reason for return if order_status is Returned; NULL otherwise. 7 possible reasons",
})

print(f"\n\u2705 All column comments applied for retail schema")

spark.sql(f"ALTER TABLE {CATALOG}.retail.customers ALTER COLUMN customer_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.retail.customers ADD CONSTRAINT pk_customers PRIMARY KEY (customer_id)")
spark.sql(f"ALTER TABLE {CATALOG}.retail.dim_product ALTER COLUMN product_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.retail.dim_product ADD CONSTRAINT pk_dim_product PRIMARY KEY (product_id)")

spark.sql(f"ALTER TABLE {CATALOG}.retail.orders ALTER COLUMN order_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.retail.orders ADD CONSTRAINT pk_orders PRIMARY KEY (order_id)")
spark.sql(f"ALTER TABLE {CATALOG}.retail.orders ADD CONSTRAINT fk_orders_customer_id FOREIGN KEY (customer_id) REFERENCES {CATALOG}.retail.customers(customer_id)")

spark.sql(f"ALTER TABLE {CATALOG}.retail.order_lines ALTER COLUMN order_line_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.retail.order_lines ALTER COLUMN order_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.retail.order_lines ALTER COLUMN product_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.retail.order_lines ADD CONSTRAINT pk_order_lines PRIMARY KEY (order_line_id)")
spark.sql(f"ALTER TABLE {CATALOG}.retail.order_lines ADD CONSTRAINT fk_order_lines_order_id FOREIGN KEY (order_id) REFERENCES {CATALOG}.retail.orders(order_id)")
spark.sql(f"ALTER TABLE {CATALOG}.retail.order_lines ADD CONSTRAINT fk_order_lines_product_id FOREIGN KEY (product_id) REFERENCES {CATALOG}.retail.dim_product(product_id)")

spark.sql(f"ALTER TABLE {CATALOG}.retail.returns ALTER COLUMN return_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.retail.returns ALTER COLUMN order_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.retail.returns ADD CONSTRAINT pk_returns PRIMARY KEY (return_id)")
spark.sql(f"ALTER TABLE {CATALOG}.retail.returns ADD CONSTRAINT fk_returns_order_id FOREIGN KEY (order_id) REFERENCES {CATALOG}.retail.orders(order_id)")
print(f"\u2714 PK/FK constraints applied for retail schema")

In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.retail') IS
'Retail sample dataset with realistic statistical distributions and Faker-generated PII. Dimension: `dim_product` (60+ products, PK: product_id). Entity: `customers` (~5K rows, PK: customer_id). Facts: `orders` (100K-500K order headers, PK: order_id, FK: customer_id), `order_lines` (1-5 lines per order, PK: order_line_id, FK: order_id → orders, product_id → dim_product), `returns` (one row per returned order, PK: return_id, FK: order_id → orders). Key features: multi-line baskets, product-level return rates, separate return events.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`retail` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"✔ RemoveAfter tag applied to retail schema ({remove_after_value})")